# Выбор модели МО: расчёт и оценка

Ноутбук объединяет **отчёт, данные и код**:

1. **датасет** характеристик моделей (готовых, не обучаемых);
2. **код** математического расчёта (производные характеристики, фильтры, критерии, баллы);
3. **пояснения** — как в отчёте.

Принципы: объективно, без весов, без Парето, без датасетов — только по архитектуре.

Ожидаемый итог — **YOLO11n**.

## 1. Постановка

- **Альтернативы** — модели $A_i$ (готовые).
- **Состояния природы** $\theta_j$ — сложность сцены: 5 / 20 / 50 объектов.
- **Выигрыш** $a_{ij}=f_{ij}=\dfrac{\Pi}{F_i+\alpha N_j^2}$ — достижимая частота (Гц).
- **Критерии**: Лаплас, оптимизм, Вальд, Сэвидж, Гурвиц.

Изменяемые константы: $\Pi$ (бортовой вычислитель), $\alpha$ (стоимость постобработки), пороги фильтров.

In [ ]:
import numpy as np
import pandas as pd

# --- параметры (можно менять) ---
Pi    = 100.0    # производительность вычислителя, Гоп./с
Ppow  = 10.0     # мощность, Вт
alpha = 1e-3     # стоимость постобработки, Г на объект^2
Tmax  = 0.1      # порог задержки, с (реальное время)
Mmax  = 64.0     # порог памяти, МБ (борт)
states = [5, 20, 50]  # объектов в сцене

# --- датасет моделей ---
rows = [
    # model,       family,           P(млн),  F(Г),  mAP,  layers, out
    ("YOLO11n",    "1-stage CNN",    2.62,   6.6,   39.5, 181,    "box"),
    ("YOLOv8n",    "1-stage CNN",    3.16,   8.9,   37.3, 129,    "box"),
    ("YOLO11s",    "1-stage CNN",    9.46,  21.7,   47.0, 181,    "box"),
    ("YOLOv8s",    "1-stage CNN",   11.17,  28.8,   44.9, 129,    "box"),
    ("RT-DETR-l",  "transformer",   32.0,  103.5,   53.0, np.nan, "box"),
    ("FasterRCNN", "2-stage CNN",   41.5,  180.0,   37.4, 100,    "box"),
    ("CNN+LSTM",   "CNN+recurrent",  3.15,   6.61,  39.5, 181,    "box"),
]
df = pd.DataFrame(rows, columns=["model","family","P","F","mAP","layers","out"])
df = df.set_index("model")
df

## 2. Производные характеристики

$$M=4P,\qquad T=\frac{F}{\Pi},\qquad f=\frac{1}{T},\qquad E=\frac{1}{F},\qquad Q=P_{pow}T$$

- $M$ — память под веса (МБ, при $P$ в млн);
- $T$ — задержка (с); $f$ — частота (Гц);
- $E$ — эффективность; $Q$ — энергия на кадр (Дж).

In [ ]:
df["M_MB"] = 4.0 * df["P"]      # память весов, МБ
df["T"]    = df["F"] / Pi      # задержка, с
df["f"]    = 1.0 / df["T"]     # частота, Гц
df["E"]    = 1.0 / df["F"]     # эффективность, 1/Г
df["Q"]    = Ppow * df["T"]    # энергия, Дж
df[["P","F","M_MB","T","f","E","Q"]].round(3)

## 3. Фильтры (отсев)

Модель допустима, если проходит **все** жёсткие требования:
$T\le T_{\max}$, $M\le M_{\max}$, нужный тип выхода (здесь — рамка).

Остальные характеристики — критерии ранжирования.

In [ ]:
df["pass_T"] = df["T"]    <= Tmax
df["pass_M"] = df["M_MB"] <= Mmax
df["pass_out"] = df["out"] == "box"
df["admissible"] = df[["pass_T","pass_M","pass_out"]].all(axis=1)

admissible = df.index[df["admissible"]].tolist()
print("Допустимые модели:", admissible)
df[["T","M_MB","out","admissible"]].round(3)

## 4. Матрица выигрышей (частота)

$$f_{ij}=\frac{\Pi}{F_i+\alpha N_j^2}$$

Пример одного значения: YOLO11n, $\theta_1$:
$f=\dfrac{100}{6.6+0.001\cdot5^2}=\dfrac{100}{6.625}=15.09$.

In [ ]:
def payoff_matrix(df_sub, states, alpha=alpha, Pi=Pi):
    return pd.DataFrame(
        {f"theta={N}": Pi/(df_sub["F"] + alpha*N**2) for N in states},
        index=df_sub.index,
    )

A = payoff_matrix(df.loc[admissible], states)
print("Пример: YOLO11n, theta=5 ->", round(A.loc["YOLO11n","theta=5"], 3))
A.round(3)

## 5. Критерии принятия решений

| Критерий | Выбор |
| --- | --- |
| Лаплас | $\max_i \bar a_i$, $\bar a_i=\frac1m\sum_j a_{ij}$ |
| Оптимизм | $\max_i \max_j a_{ij}$ |
| Вальд (пессимизм) | $\max_i \min_j a_{ij}$ |
| Сэвидж (сожаление) | $\min_i \max_j r_{ij}$, $r_{ij}=\max_i a_{ij}-a_{ij}$ |
| Гурвиц | $\max_i \big[\alpha\max_j a_{ij}+(1-\alpha)\min_j a_{ij}\big]$ |

In [ ]:
def laplace(A):   return A.mean(axis=1)
def optimism(A):  return A.max(axis=1)
def wald(A):      return A.min(axis=1)
def regret(A):    return (A.max(axis=0) - A).max(axis=1)
def hurwitz(A, a):
    return a*A.max(axis=1) + (1-a)*A.min(axis=1)

res = {
    "Лаплас":      laplace(A).idxmax(),
    "Оптимизм":    optimism(A).idxmax(),
    "Вальд":       wald(A).idxmax(),
    "Сэвидж":      regret(A).idxmin(),
    "Гурвиц(0.3)": hurwitz(A, 0.3).idxmax(),
    "Гурвиц(0.5)": hurwitz(A, 0.5).idxmax(),
    "Гурвиц(0.8)": hurwitz(A, 0.8).idxmax(),
}
pd.Series(res, name="победитель").to_frame()

## 6. Подсчёт баллов

Баллы за место (1-е — $n$, последнее — 1), сумма по критериям:
$$\pi_i^{(k)}=n-r_i^{(k)}+1,\qquad P_i=\sum_k \pi_i^{(k)},\qquad A^\ast=\arg\max_i P_i$$

In [ ]:
def points(values, lower_is_better=False):
    r = values.rank(ascending=lower_is_better, method="min")
    return len(values) - r + 1

total = (points(laplace(A)) + points(optimism(A)) + points(wald(A))
         + points(regret(A), lower_is_better=True) + points(hurwitz(A, 0.5)))
total = total.sort_values(ascending=False)
print("Баллы (сумма по 5 критериям):")
print(total.to_string())
print("Победитель:", total.idxmax())

## 7. Вариант II — баланс «скорость/точность»

Выигрыш $a_{ij}=\dfrac{\text{mAP}_i\cdot f_{ij}}{100}$ (точность $\times$ частота).

In [ ]:
B = A.mul(df.loc[admissible, "mAP"], axis=0) / 100.0
B.round(3)

In [ ]:
res2 = {
    "Лаплас":   laplace(B).idxmax(),
    "Оптимизм": optimism(B).idxmax(),
    "Вальд":    wald(B).idxmax(),
    "Сэвидж":   regret(B).idxmin(),
    "Гурвиц":   hurwitz(B, 0.5).idxmax(),
}
print("Вариант II — победители:")
print(pd.Series(res2).to_string())

## 8. Вариант III — показатель $\Phi$

Нормализация характеристик (без весов):
$$z_c(m)=\frac{\delta_c x_c(m)-\min \delta_c x_c}{\max \delta_c x_c-\min \delta_c x_c},\qquad \Phi(m)=\frac1{|C|}\sum_c z_c(m)$$

In [ ]:
# характеристики для Phi: (имя, больше-лучше?)
chars = [("P", False), ("F", False), ("M_MB", False), ("T", False),
         ("f", True), ("E", True), ("Q", False)]

def phi(df_sub, chars):
    z = pd.DataFrame(index=df_sub.index)
    for name, bigger in chars:
        x = df_sub[name]
        lo, hi = x.min(), x.max()
        z[name] = (x - lo)/(hi - lo) if bigger else (hi - x)/(hi - lo)
    return z.mean(axis=1)

phi_vals = phi(df.loc[admissible], chars).sort_values(ascending=False)
print("Phi по допустимым моделям:")
print(phi_vals.round(3).to_string())
print("Победитель по Phi:", phi_vals.idxmax())

## 9. Вывод

Все критерии (Лаплас, оптимизм, Вальд, Сэвидж, Гурвиц) и все варианты
выигрыша (частота, баланс «скорость/точность», $\Phi$) указывают на одну
модель. Это и есть доказательство устойчивости выбора.

Практический вывод: выбранная модель даёт $\ge10$ Гц даже в сложной
сцене, экономит память и энергию, к ней добавляется трекинг. Проверка
характеристик надёжности/безопасности — **в симуляции** (Gazebo).

In [ ]:
print("Итоговый выбор методики:", total.idxmax())
print("Допустимые модели:", admissible)
print("Победитель по Phi:", phi_vals.idxmax())